# 训练 DeepSeek-R1-Distill-Qwen-1.5B
我们将使用 [Unsloth](https://github.com/unslothai/unsloth) 来加速训练过程。

In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # 支持 RoPE 缩放
dtype = None # 自动检测 (True 为 Float16, False 为 Bfloat16)
load_in_4bit = True # 使用 4bit 量化减少内存使用

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 添加 LoRA 适配器
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # 秩
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 准备数据集
这里我们使用一个简单的格式示例，你可以替换为你自己的数据集。

In [ ]:
from datasets import load_dataset
import os

# 请确保你已将文件上传到 /content/ 目录下
# 如果你的文件名不同，请修改下面的路径
data_path = "alpaca.json"

if os.path.exists(data_path):
    # 加载本地 JSON 文件
    dataset = load_dataset("json", data_files=data_path, split="train")

    def formatting_prompts_func(examples):
        instructions = examples["instruction"]
        inputs       = examples.get("input", [""] * len(instructions)) # Alpaca 格式可能有 input 字段
        outputs      = examples["output"]
        texts = []
        for instruction, input_text, output in zip(instructions, inputs, outputs):
            # 合并 instruction 和 input
            full_instruction = f"{instruction}\n{input_text}" if input_text else instruction
            # 适配 DeepSeek-R1 的 Prompt 模板
            text = f"<｜begin of sentence｜>指令: {full_instruction}\n回答: {output}<｜end of sentence｜>"
            texts.append(text)
        return { "text" : texts, }

    dataset = dataset.map(formatting_prompts_func, batched = True)
    print("本地数据集加载成功！")
else:
    print(f"错误：未找到文件 {data_path}。请通过左侧侧边栏上传文件。")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 测试运行，可以增加步数
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()